In [7]:
## INSTRUCTION: DEPENDING ON WHO IS USING THIS, PLEASE COMMENT OUT THE OTHER SYS.PATH.APPEND THAT DOES NOT RELATE TO YOUR DEVICE. 

import sys
sys.path.append('/Users/annaglass/capstone/capstone')
#sys.path.append('/Users/jasmi/capstone')
#sys.path.append('/Users/moham/Downloads/New folder/capstone')
#sys.path.append('/Users/marks/OneDrive/Documents/Georgetown/capstone)
#sys.path.append('/Users\polivetti\capstone\capstone)

In [18]:
import os
print(os.getcwd())

/Users/annaglass/capstone/capstone/models/attribution


In [21]:
import pandas as pd

In [24]:
attr = pd.read_csv("../../data/attribution_all_scored.csv")
attr.head()

,kind,dimension,value,credit,credit_share,rating,rating_pct
0,item,source_feed_name,broadcast-tv,0.925621,0.232353,5,1.000000
1,item,source_feed_name,regionals-web,0.863526,0.216765,5,0.891913
2,item,source_feed_name,broadcast-radio,0.405403,0.101766,5,0.094476
3,item,source_feed_name,consumer,0.402641,0.101072,5,0.089668
4,item,source_feed_name,web,0.351127,0.088141,5,0.000000


In [25]:
attr.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11129 entries, 0 to 11128
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   kind          11129 non-null  object 
 1   dimension     11129 non-null  object 
 2   value         11129 non-null  object 
 3   credit        11129 non-null  float64
 4   credit_share  11129 non-null  float64
 5   rating        11129 non-null  int64  
 6   rating_pct    11129 non-null  float64
dtypes: float64(3), int64(1), object(3)
memory usage: 608.7+ KB


In [30]:
# Top items per dimension 
top_items = (attr.query("kind=='item'")
             .sort_values(["dimension", "credit_share"], ascending=[True,False])
             .groupby("dimension")
             .head(20))

top_items.head(20)

,kind,dimension,value,credit,credit_share,rating,rating_pct
40,item,author_name,uncredited,0.648699,0.648699,5,1.000000
41,item,author_name,__OTHER__,0.149661,0.149661,5,0.230614
42,item,author_name,Ebenezer Mensah,0.009570,0.009570,5,0.014630
43,item,author_name,Mike Stobbe,0.006053,0.006053,5,0.009208
44,item,author_name,Maria Cheng,0.003063,0.003063,5,0.004598
45,item,author_name,Robin Millard,0.002099,0.002099,5,0.003112
46,item,author_name,Brian Mann,0.001826,0.001826,5,0.002692
47,item,author_name,Meta Time,0.001676,0.001676,5,0.002460
48,item,author_name,Carla K. Johnson,0.001570,0.001570,5,0.002296
49,item,author_name,The Associated Press,0.001559,0.001559,5,0.002279


In [32]:
# Top terms overall
top_terms = (attr.query("kind=='term'")
               .sort_values("credit_share", ascending=False)
               .head(50))

top_terms.head(20)

,kind,dimension,value,credit,credit_share,rating,rating_pct
10729,term,term,illness,0.177204,0.177204,5,1.000000
10929,term,term,affected,0.147297,0.147297,5,1.000000
10730,term,term,early,0.134319,0.134319,5,0.755791
10731,term,term,available,0.112737,0.112737,5,0.632889
10732,term,term,recent,0.093094,0.093094,5,0.521032
10930,term,term,top,0.084305,0.084305,5,0.562580
10931,term,term,full,0.073392,0.073392,5,0.486799
10932,term,term,washington,0.067504,0.067504,5,0.445908
10933,term,term,trend,0.065117,0.065117,5,0.429338
10934,term,term,announced,0.062652,0.062652,5,0.412221


In [42]:
def lookup(dimension=None, value=None, term=None, k=10):
    df = attr.copy()
    if term:
        df = df.query("kind=='term' & value==@term")
    else:
        df = df.query("kind=='item' & dimension==@dimension & value==@value")
    return df.sort_values("credit_share", ascending=False).head(k)

# Examples:
print(lookup(dimension="publication_name", value="CNN"))
print(" ")
print(lookup(term="vaccine"))

      kind         dimension value    credit  credit_share  rating  rating_pct
8066  item  publication_name   CNN  0.000464      0.000464       5    0.000935
 
       kind dimension    value    credit  credit_share  rating  rating_pct
10896  term      term  vaccine  0.000018      0.000018       1    0.532547


In [43]:
# Keep each state's score (take the row per state—credit/credit_share don’t vary by content row)
item_scores = attr.query("kind=='item'")[["dimension","value","credit","credit_share","rating"]]

# Example: map publication_name credit_share onto content rows
df = pd.read_csv("/Users/annaglass/capstone/capstone/data/final_dataset_sampled.csv", low_memory=False)
df["publication_name"] = df["publication_name"].astype(str)

pub_scores = item_scores.query("dimension=='publication_name'")[["value","credit_share"]].rename(
    columns={"value":"publication_name","credit_share":"pub_credit_share"}
)
df = df.merge(pub_scores, on="publication_name", how="left").fillna({"pub_credit_share":0.0})

In [44]:
df.head()

,tag_name,article_id,source_feed_name,load_date,feed_name,author_name,source_unique_id,source_type_name,channel_name,genre,...,source_type,sentiment_band,hit_strength,vipr_weight,vipr_score,processed_headline,processed_body,headline_token_count,body_token_count,pub_credit_share
0,Public Health,18083066685,regionals-web,2024-03-25,opoint,Jamie DeLine,31229-477204,Regional News,Web,Unknown,...,Regional News,Neutral,2,685,-4795,lawmaker call independent report ny pandemic r...,albany news new york state capitol monday lost...,7,237,0.000326
1,Public Health,18083068112,regionals-web,2024-03-25,opoint,Katherine Itoh,21944-341746,Regional News,Web,Unknown,...,Regional News,Neutral,3,1300,-3900,yellowstone star forrie smith say kicked fligh...,forrie smith vocal public health belief skippe...,12,227,0.000484
2,Public Health,18083069120,trade-web,2024-03-25,opoint,uncredited,156350-188089,Trade News,Web,Unknown,...,Trade News,Neutral,13,2418,16926,free rsv immunisation program queensland infan...,joint statement premier honourable steven mile...,8,571,0.000938
3,Public Health,18083073191,broadcast-tv,2024-03-25,tveyes,uncredited,a7a238b1-8e47-43f1-b27d-db271714b0e9,TV,Broadcast,WFTX,...,TV,Negative,2,133,-5187,fox news,local news sport weather dark purple state one...,2,321,0.000000
4,Public Health,18083073192,broadcast-tv,2024-03-25,tveyes,uncredited,36c13958-e8cf-4b90-9713-4c3838eed8f4,TV,Broadcast,WFTXBACK,...,TV,Negative,2,133,-5187,fox news,local news sport weather dark purple state one...,2,321,0.000000


In [45]:
term_scores = (attr.query("kind=='term'")
                 [["value","credit_share"]]
                 .drop_duplicates()
                 .rename(columns={"value":"term","credit_share":"term_credit_share"}))

# Simple binary presence → max term influence on row
def max_term_influence(headline, body):
    txt = f"{str(headline or '')} {str(body or '')}".lower()
    hits = term_scores["term"].loc[term_scores["term"].map(lambda t: t in txt)]
    if hits.empty: return 0.0
    return term_scores.set_index("term").loc[hits, "term_credit_share"].max()

df["max_term_credit"] = df.apply(lambda r: max_term_influence(r.get("processed_headline"), r.get("processed_body")), axis=1)

In [48]:
df.head()

,tag_name,article_id,source_feed_name,load_date,feed_name,author_name,source_unique_id,source_type_name,channel_name,genre,...,sentiment_band,hit_strength,vipr_weight,vipr_score,processed_headline,processed_body,headline_token_count,body_token_count,pub_credit_share,max_term_credit
0,Public Health,18083066685,regionals-web,2024-03-25,opoint,Jamie DeLine,31229-477204,Regional News,Web,Unknown,...,Neutral,2,685,-4795,lawmaker call independent report ny pandemic r...,albany news new york state capitol monday lost...,7,237,0.000326,0.056167
1,Public Health,18083068112,regionals-web,2024-03-25,opoint,Katherine Itoh,21944-341746,Regional News,Web,Unknown,...,Neutral,3,1300,-3900,yellowstone star forrie smith say kicked fligh...,forrie smith vocal public health belief skippe...,12,227,0.000484,0.084305
2,Public Health,18083069120,trade-web,2024-03-25,opoint,uncredited,156350-188089,Trade News,Web,Unknown,...,Neutral,13,2418,16926,free rsv immunisation program queensland infan...,joint statement premier honourable steven mile...,8,571,0.000938,0.177204
3,Public Health,18083073191,broadcast-tv,2024-03-25,tveyes,uncredited,a7a238b1-8e47-43f1-b27d-db271714b0e9,TV,Broadcast,WFTX,...,Negative,2,133,-5187,fox news,local news sport weather dark purple state one...,2,321,0.000000,0.134319
4,Public Health,18083073192,broadcast-tv,2024-03-25,tveyes,uncredited,36c13958-e8cf-4b90-9713-4c3838eed8f4,TV,Broadcast,WFTXBACK,...,Negative,2,133,-5187,fox news,local news sport weather dark purple state one...,2,321,0.000000,0.134319


In [47]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 379225 entries, 0 to 379224
Data columns (total 25 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   tag_name              379225 non-null  object 
 1   article_id            379225 non-null  int64  
 2   source_feed_name      379225 non-null  object 
 3   load_date             379225 non-null  object 
 4   feed_name             379225 non-null  object 
 5   author_name           379225 non-null  object 
 6   source_unique_id      379225 non-null  object 
 7   source_type_name      379225 non-null  object 
 8   channel_name          379225 non-null  object 
 9   genre                 379225 non-null  object 
 10  publisher_name        379225 non-null  object 
 11  publication_name      379225 non-null  object 
 12  circulation_size      379225 non-null  int64  
 13  sentiment_score       379225 non-null  int64  
 14  source_type           379225 non-null  object 
 15  

In [51]:
from pathlib import Path

ROOT = Path("/Users/annaglass/capstone/capstone/data")
out_path = ROOT / "final_model_dataset.csv"
df.to_csv(out_path, index=False)

In [52]:
fdf = pd.read_csv("../../data/final_model_dataset.csv")

In [53]:
fdf.head()

,tag_name,article_id,source_feed_name,load_date,feed_name,author_name,source_unique_id,source_type_name,channel_name,genre,...,sentiment_band,hit_strength,vipr_weight,vipr_score,processed_headline,processed_body,headline_token_count,body_token_count,pub_credit_share,max_term_credit
0,Public Health,18083066685,regionals-web,2024-03-25,opoint,Jamie DeLine,31229-477204,Regional News,Web,Unknown,...,Neutral,2,685,-4795,lawmaker call independent report ny pandemic r...,albany news new york state capitol monday lost...,7,237,0.000326,0.056167
1,Public Health,18083068112,regionals-web,2024-03-25,opoint,Katherine Itoh,21944-341746,Regional News,Web,Unknown,...,Neutral,3,1300,-3900,yellowstone star forrie smith say kicked fligh...,forrie smith vocal public health belief skippe...,12,227,0.000484,0.084305
2,Public Health,18083069120,trade-web,2024-03-25,opoint,uncredited,156350-188089,Trade News,Web,Unknown,...,Neutral,13,2418,16926,free rsv immunisation program queensland infan...,joint statement premier honourable steven mile...,8,571,0.000938,0.177204
3,Public Health,18083073191,broadcast-tv,2024-03-25,tveyes,uncredited,a7a238b1-8e47-43f1-b27d-db271714b0e9,TV,Broadcast,WFTX,...,Negative,2,133,-5187,fox news,local news sport weather dark purple state one...,2,321,0.000000,0.134319
4,Public Health,18083073192,broadcast-tv,2024-03-25,tveyes,uncredited,36c13958-e8cf-4b90-9713-4c3838eed8f4,TV,Broadcast,WFTXBACK,...,Negative,2,133,-5187,fox news,local news sport weather dark purple state one...,2,321,0.000000,0.134319


In [54]:
fdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 379225 entries, 0 to 379224
Data columns (total 25 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   tag_name              379225 non-null  object 
 1   article_id            379225 non-null  int64  
 2   source_feed_name      379225 non-null  object 
 3   load_date             379225 non-null  object 
 4   feed_name             379225 non-null  object 
 5   author_name           379225 non-null  object 
 6   source_unique_id      379225 non-null  object 
 7   source_type_name      379225 non-null  object 
 8   channel_name          379225 non-null  object 
 9   genre                 379225 non-null  object 
 10  publisher_name        379225 non-null  object 
 11  publication_name      379225 non-null  object 
 12  circulation_size      379225 non-null  int64  
 13  sentiment_score       379225 non-null  int64  
 14  source_type           379225 non-null  object 
 15  

In [2]:
from azure.storage.blob import BlobServiceClient
import os

conn = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
svc = BlobServiceClient.from_connection_string(conn)
print("OK, account:", svc.account_name)
for c in svc.list_containers(name_starts_with=""):
    print("container:", c["name"])


/Users/annaglass/capstone/capstone/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


OK, account: pentacapstonefiles
container: capstone
